# RNAAPIA — Face Recognition v7
**Ponto de partida:** `best_cnn.pth` e `best_resnet.pth` (90.02% e 90.24%)

**Melhorias em relação ao v5:**
- Mesmas 50 identidades do v5 (via dataset_metadata.json)
- Label Smoothing (0.1) na Cross-Entropy
- Learning rate mais baixo (1e-4) — treino mais fino a partir dos pesos já aprendidos
- Weight decay mais alto (1e-3)
- Dropout aumentado para 0.5
- RandomErasing no augmentation
- 50 épocas (mais margem para convergir)

```
0. Setup + localizar checkpoints
1. MTCNN — confirmar dataset alinhado
2. Filtragem + equalização
3. DataLoaders
4. Modelos + carregar pesos
5. Treino FaceCNN (continua dos pesos anteriores)
6. Treino ResNet-50 (continua dos pesos anteriores)
7. Comparação v5 vs v7
8. Fine-tuning organizacional
9. Avaliação FAR/FRR/EER
10. Script de inferência local
```

## ⚠️ Antes de começar
1. **Add Data** → `hearfool/vggface2`
2. **Add Data** → Models → `facecnn` e `resnet-50`
3. **Add Data** → `dataset-metadata-json`
4. **Settings** → Accelerator → GPU T4
5. Corre as células por ordem

---
## 0. Setup

In [8]:
import torch, os, glob

print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  Sem GPU! Ativa em Settings → Accelerator → GPU')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CUDA disponível: True
GPU: Tesla T4
VRAM: 15.6 GB


In [10]:
BASE = '/kaggle/working'

PATHS = {
    'raw'         : '/kaggle/input/datasets/hearfool/vggface2/train',
    'aligned'     : f'{BASE}/vggface2_aligned',
    'org_dataset' : f'{BASE}/org_dataset',
    'org_aligned' : f'{BASE}/org_aligned',
    'checkpoints' : f'{BASE}/checkpoints',
    'logs'        : f'{BASE}/logs',
}
for k, v in PATHS.items():
    if k != 'raw':
        os.makedirs(v, exist_ok=True)

# Paths diretos para os ficheiros do v5
CNN_CKPT_IN    = '/kaggle/input/models/goncalojesus/facecnn/pytorch/default/1/best_cnn.pth'
RESNET_CKPT_IN = '/kaggle/input/models/goncalojesus/resnet-50/pytorch/default/1/best_resnet.pth'
METADATA_PATH  = '/kaggle/input/datasets/goncalojesus/dataset-metadata-json/dataset_metadata.json'

# Verificar
print('A verificar ficheiros:')
for name, path in [('CNN', CNN_CKPT_IN), ('ResNet', RESNET_CKPT_IN), ('Metadata', METADATA_PATH)]:
    status = '✓' if os.path.exists(path) else '⚠️  NÃO ENCONTRADO'
    print(f'  {status} {name}: {path}')

# Hiperparâmetros
IMAGE_SIZE     = 192
EMBEDDING_SIZE = 512
DROPOUT        = 0.5   # era 0.4
N_IDENTITIES   = 75    # 50 anteriores + 25 novas
EPOCHS         = 50    # era 30
LR             = 1e-4  # era 1e-3
WEIGHT_DECAY   = 1e-3  # era 1e-4
LABEL_SMOOTH   = 0.1
BATCH_SIZE     = 64

# Referência do v5
PREV_BEST_CNN    = 0.9098  # melhor CNN v8 (90.98% com 70 identidades)
PREV_BEST_RESNET = 0.9460  # melhor ResNet v7 (ResNet v8 ficou a meio — epoch 6)

print(f'\nHiperparâmetros v7:')
print(f'  N_IDENTITIES={N_IDENTITIES} | EPOCHS={EPOCHS} | LR={LR} | WD={WEIGHT_DECAY}')
print(f'  DROPOUT={DROPOUT} | LABEL_SMOOTH={LABEL_SMOOTH} | IMAGE_SIZE={IMAGE_SIZE}')

A verificar ficheiros:
  ✓ CNN: /kaggle/input/models/goncalojesus/facecnn/pytorch/default/1/best_cnn.pth
  ✓ ResNet: /kaggle/input/models/goncalojesus/resnet-50/pytorch/default/1/best_resnet.pth
  ✓ Metadata: /kaggle/input/datasets/goncalojesus/dataset-metadata-json/dataset_metadata.json

Hiperparâmetros v7:
  N_IDENTITIES=75 | EPOCHS=50 | LR=0.0001 | WD=0.001
  DROPOUT=0.5 | LABEL_SMOOTH=0.1 | IMAGE_SIZE=192


In [11]:
%%capture
!pip install facenet-pytorch==2.5.3 tqdm matplotlib scikit-learn
print('✓ Dependências instaladas')

---
## 1. MTCNN — Alinhar 75 Identidades
As 50 identidades anteriores já estão alinhadas — só processa as 25 novas.
Retoma automaticamente se a sessão cair.

In [13]:
import json, shutil
import numpy as np
from PIL import Image
from tqdm import tqdm
from facenet_pytorch import MTCNN

all_identities = sorted(os.listdir(PATHS['raw']))
print(f'Identidades disponíveis no VGGFace2: {len(all_identities)}')

# Carregar as 50 identidades anteriores + adicionar 25 novas
if METADATA_PATH and os.path.exists(METADATA_PATH):
    with open(METADATA_PATH) as f:
        metadata = json.load(f)
    prev_ids = metadata['identities']
    new_ids  = [i for i in all_identities if i not in prev_ids][:25]
    identities_to_use = prev_ids + new_ids
    print(f'  ✓ {len(prev_ids)} anteriores + {len(new_ids)} novas = {len(identities_to_use)} total')
else:
    identities_to_use = all_identities[:N_IDENTITIES]
    print(f'  Sem metadata — a usar primeiras {N_IDENTITIES} identidades')

# Verificar quais precisam de ser (re)processadas
# Uma identidade é reprocessada se não existe ou tem menos de MAX_IMAGES_PER_IDENTITY imagens
MAX_IMAGES_PER_IDENTITY = 350
to_process = []
for identity in identities_to_use:
    id_path = os.path.join(PATHS['aligned'], identity)
    if not os.path.exists(id_path):
        to_process.append((identity, 'nova'))
    else:
        n_existing = len(os.listdir(id_path))
        if n_existing < MAX_IMAGES_PER_IDENTITY * 0.8:  # reprocessa se tiver menos de 80% do máximo
            to_process.append((identity, f'incompleta ({n_existing} imgs)'))

print(f'  Por processar/reprocessar: {len(to_process)}')
for name, reason in to_process[:10]:
    print(f'    {name}: {reason}')
if len(to_process) > 10:
    print(f'    ... e mais {len(to_process)-10}')

Identidades disponíveis no VGGFace2: 480
  ✓ 50 anteriores + 25 novas = 75 total
  Por processar/reprocessar: 75
    n000002: nova
    n000003: incompleta (50 imgs)
    n000004: incompleta (50 imgs)
    n000005: incompleta (50 imgs)
    n000006: incompleta (50 imgs)
    n000007: incompleta (50 imgs)
    n000008: incompleta (50 imgs)
    n000010: incompleta (50 imgs)
    n000011: incompleta (50 imgs)
    n000012: incompleta (50 imgs)
    ... e mais 65


In [14]:
# Só corre se houver identidades por processar
if to_process:
    MIN_IMAGES_PER_IDENTITY = 20
    margin = max(20, IMAGE_SIZE // 6)

    mtcnn = MTCNN(
        image_size=IMAGE_SIZE, margin=margin,
        min_face_size=20, thresholds=[0.6, 0.7, 0.7],
        factor=0.709, post_process=False, device=device
    )

    total_saved, failed = 0, 0
    for identity, reason in tqdm(to_process, desc='MTCNN'):
        out_dir = os.path.join(PATHS['aligned'], identity)
        # Limpar pasta se já existia (reprocessar)
        if os.path.exists(out_dir):
            shutil.rmtree(out_dir)
        os.makedirs(out_dir, exist_ok=True)
        img_paths = (glob.glob(os.path.join(PATHS['raw'], identity, '*.jpg')) +
                     glob.glob(os.path.join(PATHS['raw'], identity, '*.png')))[:MAX_IMAGES_PER_IDENTITY]
        saved = 0
        for img_path in img_paths:
            try:
                face = mtcnn(Image.open(img_path).convert('RGB'))
                if face is not None:
                    Image.fromarray(face.permute(1,2,0).byte().numpy()).save(
                        os.path.join(out_dir, os.path.basename(img_path)))
                    saved += 1
            except Exception:
                failed += 1
        if saved < MIN_IMAGES_PER_IDENTITY:
            shutil.rmtree(out_dir)
        else:
            total_saved += saved

    print(f'✓ MTCNN concluído: {len(os.listdir(PATHS["aligned"]))} identidades | {total_saved} novas imagens | {failed} falhas')
else:
    # Inicializar mtcnn na mesma (necessário para secção 8)
    margin = max(20, IMAGE_SIZE // 6)
    mtcnn  = MTCNN(image_size=IMAGE_SIZE, margin=margin,
                   min_face_size=20, post_process=False, device=device)
    print('✓ Todas as identidades já processadas com imagens suficientes')
    print('  MTCNN inicializado para uso posterior (secção 8)')

total_imgs = sum(len(os.listdir(os.path.join(PATHS['aligned'], d)))
                 for d in os.listdir(PATHS['aligned']))
print(f'  Total imagens no dataset: {total_imgs}')

MTCNN: 100%|██████████| 75/75 [16:49<00:00, 13.46s/it]

✓ MTCNN concluído: 75 identidades | 23578 novas imagens | 0 falhas
  Total imagens no dataset: 23578


---
## 2. Filtragem + Equalização

In [16]:
import random
import matplotlib.pyplot as plt

id_counts = {d: len(os.listdir(os.path.join(PATHS['aligned'], d)))
             for d in os.listdir(PATHS['aligned'])}
counts = np.array(list(id_counts.values()))

print(f'Identidades: {len(counts)} | Total imagens: {counts.sum()}')
print(f'Por identidade — min: {counts.min()} | mediana: {int(np.median(counts))} | max: {counts.max()}')

# Remover outliers (IQR)
Q1, Q3 = np.percentile(counts, 25), np.percentile(counts, 75)
IQR    = Q3 - Q1
lower  = max(MIN_IMAGES_PER_IDENTITY, Q1 - 1.5 * IQR)
upper  = Q3 + 1.5 * IQR
removed = 0
for identity, n in id_counts.items():
    if n < lower or n > upper:
        shutil.rmtree(os.path.join(PATHS['aligned'], identity))
        removed += 1
print(f'Outliers removidos: {removed}')

# Equalizar ao máximo da mediana
random.seed(42)
MAX_PER_ID = int(np.median([len(os.listdir(os.path.join(PATHS['aligned'], d)))
                            for d in os.listdir(PATHS['aligned'])]))
for identity in os.listdir(PATHS['aligned']):
    id_path = os.path.join(PATHS['aligned'], identity)
    imgs    = os.listdir(id_path)
    if len(imgs) > MAX_PER_ID:
        for f in random.sample(imgs, len(imgs) - MAX_PER_ID):
            os.remove(os.path.join(id_path, f))

final_ids  = os.listdir(PATHS['aligned'])
final_imgs = sum(len(os.listdir(os.path.join(PATHS['aligned'], d))) for d in final_ids)
print(f'\n✓ Dataset final: {len(final_ids)} identidades | {final_imgs} imagens | máx {MAX_PER_ID}/id')

# Guardar metadata atualizado
new_metadata = {
    'image_size': IMAGE_SIZE,
    'num_identities': len(final_ids),
    'num_images': final_imgs,
    'max_per_id': MAX_PER_ID,
    'version': 'v7',
    'identities': sorted(final_ids)
}
with open(f'{BASE}/dataset_metadata_v7.json', 'w') as f:
    json.dump(new_metadata, f, indent=2)
print('  ✓ dataset_metadata_v7.json guardado')

Identidades: 75 | Total imagens: 23578
Por identidade — min: 156 | mediana: 346 | max: 350
Outliers removidos: 5

✓ Dataset final: 70 identidades | 22544 imagens | máx 348/id
  ✓ dataset_metadata_v7.json guardado


---
## 3. DataLoaders com Augmentation Melhorado

In [17]:
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.1),
    transforms.RandomApply([transforms.RandomRotation(15)], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),  # novo
])
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

full_dataset = datasets.ImageFolder(PATHS['aligned'], transform=train_transform)
NUM_CLASSES  = len(full_dataset.classes)
total        = len(full_dataset)
n_train      = int(0.70 * total)
n_val        = int(0.15 * total)
n_test       = total - n_train - n_val

train_set, val_set, test_set = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Classes: {NUM_CLASSES} | Total: {total}')
print(f'Train: {n_train} | Val: {n_val} | Test: {n_test}')
print(f'Batches por época: {len(train_loader)}')

Classes: 70 | Total: 22544
Train: 15780 | Val: 3381 | Test: 3383
Batches por época: 247


---
## 4. Modelos + Carregar Pesos do v5

In [18]:
import torch.nn as nn
import torchvision.models as tv_models

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, pool=2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, padding=kernel//2, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True), nn.MaxPool2d(pool))
    def forward(self, x): return self.block(x)

class FaceCNN(nn.Module):
    def __init__(self, num_classes, embedding_size=512, image_size=192, dropout=0.5):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            ConvBlock(3, 32), ConvBlock(32, 64),
            ConvBlock(64, 128), ConvBlock(128, 256))
        fm        = image_size // 16
        flat_size = 256 * fm * fm
        self.embedding = nn.Sequential(
            nn.Flatten(), nn.Dropout(p=dropout),
            nn.Linear(flat_size, embedding_size),
            nn.BatchNorm1d(embedding_size), nn.ReLU(inplace=True))
        self.classifier = nn.Linear(embedding_size, num_classes)
    def forward(self, x, return_embedding=False):
        emb = self.embedding(self.conv_blocks(x))
        return F.normalize(emb, dim=1) if return_embedding else self.classifier(emb)

class ResNet50Face(nn.Module):
    def __init__(self, num_classes, embedding_size=512, dropout=0.5):
        super().__init__()
        backbone       = tv_models.resnet50(weights=None)
        self.features  = nn.Sequential(*list(backbone.children())[:-1])
        self.embedding = nn.Sequential(
            nn.Flatten(), nn.Dropout(p=dropout),
            nn.Linear(2048, embedding_size),
            nn.BatchNorm1d(embedding_size), nn.ReLU(inplace=True))
        self.classifier = nn.Linear(embedding_size, num_classes)
    def forward(self, x, return_embedding=False):
        emb = self.embedding(self.features(x))
        return F.normalize(emb, dim=1) if return_embedding else self.classifier(emb)


# Instanciar com NUM_CLASSES atual (100 identidades)
cnn_model    = FaceCNN(NUM_CLASSES, EMBEDDING_SIZE, IMAGE_SIZE, DROPOUT).to(device)
resnet_model = ResNet50Face(NUM_CLASSES, EMBEDDING_SIZE, DROPOUT).to(device)

# Carregar pesos do v5 — apenas conv_blocks + embedding (o classifier muda pois temos mais classes)
def load_partial(model, ckpt_path, device):
    if not ckpt_path:
        print('  Sem checkpoint — a treinar do zero')
        return
    ckpt       = torch.load(ckpt_path, map_location=device)
    state      = ckpt['model']
    model_dict = model.state_dict()
    # Carregar tudo exceto o classifier (que tem dimensão diferente)
    to_load = {k: v for k, v in state.items()
               if k in model_dict and 'classifier' not in k
               and v.shape == model_dict[k].shape}
    model_dict.update(to_load)
    model.load_state_dict(model_dict)
    print(f'  ✓ {len(to_load)}/{len(model_dict)} camadas carregadas | classifier reiniciado para {NUM_CLASSES} classes')

print('FaceCNN:')
load_partial(cnn_model, CNN_CKPT_IN, device)
print('ResNet-50:')
load_partial(resnet_model, RESNET_CKPT_IN, device)

print(f'\n✓ FaceCNN    — {sum(p.numel() for p in cnn_model.parameters())/1e6:.2f}M params')
print(f'✓ ResNet50   — {sum(p.numel() for p in resnet_model.parameters())/1e6:.1f}M params')

FaceCNN:
  ✓ 31/33 camadas carregadas | classifier reiniciado para 70 classes
ResNet-50:
  ✓ 325/327 camadas carregadas | classifier reiniciado para 70 classes

✓ FaceCNN    — 19.30M params
✓ ResNet50   — 24.6M params


---
## 5. Treino FaceCNN

In [23]:
import time, json
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

def train_one_epoch(model, loader, optimizer, scaler, device, label_smoothing=0.1):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, desc='  Train', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            out  = model(imgs)
            loss = F.cross_entropy(out, labels, label_smoothing=label_smoothing)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, desc='  Val', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        out  = model(imgs).float()
        loss = F.cross_entropy(out, labels)
        if not torch.isnan(loss):
            total_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total   += imgs.size(0)
    return total_loss / total, correct / total

def save_checkpoint(path, epoch, model, optimizer, scheduler, history):
    torch.save({'epoch': epoch, 'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'history': history}, path)

print('✓ Funções prontas')

✓ Funções prontas


In [24]:
from tqdm import tqdm

cnn_optimizer = Adam(cnn_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
cnn_scheduler = CosineAnnealingLR(cnn_optimizer, T_max=EPOCHS, eta_min=1e-7)
cnn_scaler    = torch.amp.GradScaler('cuda')
cnn_hist      = []
best_cnn_acc  = 0.0

CNN_LATEST = f"{PATHS['checkpoints']}/cnn_v7_latest.pth"
CNN_BEST   = f'{BASE}/best_cnn_v7.pth'
CNN_LOG    = f"{PATHS['logs']}/cnn_v7_log.json"

# Retomar se sessão cair
if os.path.exists(CNN_LATEST):
    ckpt         = torch.load(CNN_LATEST, map_location=device)
    cnn_model.load_state_dict(ckpt['model'])
    cnn_optimizer.load_state_dict(ckpt['optimizer'])
    cnn_scheduler.load_state_dict(ckpt['scheduler'])
    cnn_hist     = ckpt.get('history', [])
    START_CNN    = ckpt['epoch'] + 1
    best_cnn_acc = max((h['val_acc'] for h in cnn_hist), default=0.0)
    print(f'✓ A retomar CNN v7 do epoch {START_CNN} | melhor até agora: {best_cnn_acc*100:.2f}%')
else:
    START_CNN = 0
    print(f'A iniciar CNN v7 — {EPOCHS} épocas | {NUM_CLASSES} classes')

for epoch in range(START_CNN, EPOCHS):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(cnn_model, train_loader, cnn_optimizer, cnn_scaler, device, LABEL_SMOOTH)
    val_loss,   val_acc   = evaluate(cnn_model, val_loader, device)
    cnn_scheduler.step()
    elapsed = time.time() - t0

    record = {'epoch': epoch, 'train_loss': round(train_loss,4), 'train_acc': round(train_acc,4),
              'val_loss': round(val_loss,4), 'val_acc': round(val_acc,4), 'time_s': round(elapsed,1)}
    cnn_hist.append(record)

    improved = ' ✓' if val_acc > best_cnn_acc else ''
    print(f'[CNN v7] {epoch+1:02d}/{EPOCHS} | '
          f'Train: {train_loss:.4f} ({train_acc*100:.1f}%) | '
          f'Val: {val_loss:.4f} ({val_acc*100:.1f}%){improved} | {elapsed:.0f}s')

    save_checkpoint(CNN_LATEST, epoch, cnn_model, cnn_optimizer, cnn_scheduler, cnn_hist)
    if val_acc > best_cnn_acc:
        best_cnn_acc = val_acc
        if val_acc > PREV_BEST_CNN:  # só guarda se superar o v5
            save_checkpoint(CNN_BEST, epoch, cnn_model, cnn_optimizer, cnn_scheduler, cnn_hist)
            print(f'  ✓ Novo melhor CNN guardado ({val_acc*100:.2f}%) — superou v5 ({PREV_BEST_CNN*100:.2f}%)')
    with open(CNN_LOG, 'w') as f:
        json.dump(cnn_hist, f, indent=2)

print(f'\n✓ CNN v7 concluída!')
delta = (best_cnn_acc - PREV_BEST_CNN) * 100
print(f'  v5: {PREV_BEST_CNN*100:.2f}% → v7: {best_cnn_acc*100:.2f}% ({delta:+.2f}%)')

✓ A retomar CNN v7 do epoch 20 | melhor até agora: 89.68%


[CNN v7] 21/50 | Train: 0.9568 (98.3%) | Val: 0.6388 (89.4%) | 38s


[CNN v7] 22/50 | Train: 0.9509 (98.3%) | Val: 0.6326 (89.5%) | 38s


[CNN v7] 23/50 | Train: 0.9485 (98.4%) | Val: 0.6416 (89.4%) | 38s


[CNN v7] 24/50 | Train: 0.9416 (98.5%) | Val: 0.6427 (89.7%) | 37s


...

[CNN v7] 46/50 | Train: 0.8815 (99.4%) | Val: 0.6263 (90.6%) | 37s


[CNN v7] 47/50 | Train: 0.8808 (99.4%) | Val: 0.6255 (90.9%) | 37s


[CNN v7] 48/50 | Train: 0.8815 (99.3%) | Val: 0.6270 (90.7%) | 38s


[CNN v7] 49/50 | Train: 0.8787 (99.4%) | Val: 0.6253 (90.8%) | 38s


[CNN v7] 50/50 | Train: 0.8828 (99.3%) | Val: 0.6115 (91.0%) | 37s

✓ CNN v7 concluída!
  v5: 90.06% → v7: 90.98% (+0.92%)


---
## 6. Treino ResNet-50

In [ ]:
rn_optimizer = Adam(resnet_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
rn_scheduler = CosineAnnealingLR(rn_optimizer, T_max=EPOCHS, eta_min=1e-7)
rn_scaler    = torch.amp.GradScaler('cuda')
rn_hist      = []
best_rn_acc  = 0.0

RN_LATEST = f"{PATHS['checkpoints']}/resnet_v7_latest.pth"
RN_BEST   = f'{BASE}/best_resnet_v7.pth'
RN_LOG    = f"{PATHS['logs']}/resnet_v7_log.json"

if os.path.exists(RN_LATEST):
    ckpt         = torch.load(RN_LATEST, map_location=device)
    resnet_model.load_state_dict(ckpt['model'])
    rn_optimizer.load_state_dict(ckpt['optimizer'])
    rn_scheduler.load_state_dict(ckpt['scheduler'])
    rn_hist     = ckpt.get('history', [])
    START_RN    = ckpt['epoch'] + 1
    best_rn_acc = max((h['val_acc'] for h in rn_hist), default=0.0)
    print(f'✓ A retomar ResNet v7 do epoch {START_RN} | melhor até agora: {best_rn_acc*100:.2f}%')
else:
    START_RN = 0
    print(f'A iniciar ResNet v8 — {EPOCHS} épocas | {NUM_CLASSES} classes')
    print(f'  (sessão anterior ficou no epoch 6 — a retomar do checkpoint se existir)')

for epoch in range(START_RN, EPOCHS):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(resnet_model, train_loader, rn_optimizer, rn_scaler, device, LABEL_SMOOTH)
    val_loss,   val_acc   = evaluate(resnet_model, val_loader, device)
    rn_scheduler.step()
    elapsed = time.time() - t0

    record = {'epoch': epoch, 'train_loss': round(train_loss,4), 'train_acc': round(train_acc,4),
              'val_loss': round(val_loss,4), 'val_acc': round(val_acc,4), 'time_s': round(elapsed,1)}
    rn_hist.append(record)

    improved = ' ✓' if val_acc > best_rn_acc else ''
    print(f'[ResNet v7] {epoch+1:02d}/{EPOCHS} | '
          f'Train: {train_loss:.4f} ({train_acc*100:.1f}%) | '
          f'Val: {val_loss:.4f} ({val_acc*100:.1f}%){improved} | {elapsed:.0f}s')

    save_checkpoint(RN_LATEST, epoch, resnet_model, rn_optimizer, rn_scheduler, rn_hist)
    if val_acc > best_rn_acc:
        best_rn_acc = val_acc
        if val_acc > PREV_BEST_RESNET:  # só guarda se superar o v5
            save_checkpoint(RN_BEST, epoch, resnet_model, rn_optimizer, rn_scheduler, rn_hist)
            print(f'  ✓ Novo melhor ResNet guardado ({val_acc*100:.2f}%) — superou v5 ({PREV_BEST_RESNET*100:.2f}%)')
    with open(RN_LOG, 'w') as f:
        json.dump(rn_hist, f, indent=2)

print(f'\n✓ ResNet v7 concluída!')
delta = (best_rn_acc - PREV_BEST_RESNET) * 100
print(f'  v5: {PREV_BEST_RESNET*100:.2f}% → v7: {best_rn_acc*100:.2f}% ({delta:+.2f}%)')

A iniciar ResNet v7 — 50 épocas | 70 classes


[ResNet v7] 01/50 | Train: 2.9766 (48.0%) | Val: 1.7831 (74.5%) ✓ | 59s


[ResNet v7] 02/50 | Train: 1.7974 (76.5%) | Val: 1.1233 (80.1%) ✓ | 58s


[ResNet v7] 03/50 | Train: 1.4732 (82.8%) | Val: 0.9520 (80.7%) ✓ | 58s


[ResNet v7] 04/50 | Train: 1.3387 (85.9%) | Val: 0.8302 (82.4%) ✓ | 58s


[ResNet v7] 05/50 | Train: 1.2582 (88.2%) | Val: 0.7626 (84.2%) ✓ | 58s


[ResNet v7] 06/50 | Train: 1.2006 (89.6%) | Val: 0.7361 (84.1%) | 58s


  Train:  54%|█████▍    | 133/247 [00:26<00:22,  5.00it/s]

---
## 7. Comparação v5 vs v7

In [21]:
with open(CNN_LOG) as f:  cnn_hist = json.load(f)
with open(RN_LOG)  as f:  rn_hist  = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, key, ylabel in [
    (axes[0], 'val_loss', 'Cross-Entropy Loss'),
    (axes[1], 'val_acc',  'Accuracy (%)'),
]:
    scale = 100 if key == 'val_acc' else 1
    ax.plot([h['epoch']+1 for h in cnn_hist], [h[key]*scale for h in cnn_hist],
            'b-o', markersize=3, label='FaceCNN v7')
    ax.plot([h['epoch']+1 for h in rn_hist],  [h[key]*scale for h in rn_hist],
            'r-o', markersize=3, label='ResNet-50 v7')
    if key == 'val_acc':
        ax.axhline(PREV_BEST_CNN*100,    color='blue', linestyle='--', alpha=0.4, label=f'CNN v5: {PREV_BEST_CNN*100:.2f}%')
        ax.axhline(PREV_BEST_RESNET*100, color='red',  linestyle='--', alpha=0.4, label=f'ResNet v5: {PREV_BEST_RESNET*100:.2f}%')
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle(f'v7 — {NUM_CLASSES} identidades | Label Smoothing 0.1 | Dropout 0.5 | LR 1e-4 | 50 épocas', fontsize=12)
plt.tight_layout()
plt.savefig(f'{BASE}/comparison_v7.png', dpi=150, bbox_inches='tight')
plt.show()

best_cnn_r = max(cnn_hist, key=lambda h: h['val_acc'])
best_rn_r  = max(rn_hist,  key=lambda h: h['val_acc'])

print('\n── Resumo ─────────────────────────────────────────────────────')
print(f'{"":28s} {"FaceCNN":>10s} {"ResNet-50":>10s}')
print(f'{"v5 (baseline)":28s} {PREV_BEST_CNN*100:>9.2f}% {PREV_BEST_RESNET*100:>9.2f}%')
print(f'{"v7 (esta sessão)":28s} {best_cnn_r["val_acc"]*100:>9.2f}% {best_rn_r["val_acc"]*100:>9.2f}%')
print(f'{"Diferença":28s} {(best_cnn_r["val_acc"]-PREV_BEST_CNN)*100:>+9.2f}% {(best_rn_r["val_acc"]-PREV_BEST_RESNET)*100:>+9.2f}%')
cnn_gap = (best_cnn_r['train_acc'] - best_cnn_r['val_acc']) * 100
rn_gap  = (best_rn_r['train_acc']  - best_rn_r['val_acc'])  * 100
print(f'{"Gap train/val (overfitting)":28s} {cnn_gap:>+9.2f}% {rn_gap:>+9.2f}%')
print(f'  (v5 tinha ~9.5% — quanto mais baixo, menos overfitting)')

print('-----------------------------------')
# Métricas detalhadas — avaliação no test set
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, top_k_accuracy_score, f1_score
)
import matplotlib.pyplot as plt
import numpy as np

def get_predictions(model, loader, device):
    """Devolve labels reais, labels previstos e probabilidades."""
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            out  = model(imgs).float().cpu()
            probs = torch.softmax(out, dim=1)
            all_labels.append(labels)
            all_preds.append(out.argmax(dim=1))
            all_probs.append(probs)
    return (
        torch.cat(all_labels).numpy(),
        torch.cat(all_preds).numpy(),
        torch.cat(all_probs).numpy()
    )

# Carregar melhores modelos
ckpt = torch.load(CNN_BEST, map_location=device)
cnn_model.load_state_dict(ckpt['model'])
ckpt = torch.load(RN_BEST, map_location=device)
resnet_model.load_state_dict(ckpt['model'])

y_true_cnn,  y_pred_cnn,  y_prob_cnn  = get_predictions(cnn_model,    test_loader, device)
y_true_rn,   y_pred_rn,   y_prob_rn   = get_predictions(resnet_model, test_loader, device)

# ── Accuracy, F1, Top-5, AUC ─────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_prob, name):
    acc    = (y_true == y_pred).mean() * 100
    f1_mac = f1_score(y_true, y_pred, average='macro',    zero_division=0) * 100
    f1_w   = f1_score(y_true, y_pred, average='weighted', zero_division=0) * 100
    top5   = top_k_accuracy_score(y_true, y_prob, k=5)   * 100
    auc    = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro') * 100
    print(f'\n── {name} ──────────────────────────────')
    print(f'  Accuracy:          {acc:.2f}%')
    print(f'  F1 Macro:          {f1_mac:.2f}%')
    print(f'  F1 Weighted:       {f1_w:.2f}%')
    print(f'  Top-5 Accuracy:    {top5:.2f}%')
    print(f'  AUC (OvR macro):   {auc:.2f}%')
    return {'acc': acc, 'f1_mac': f1_mac, 'f1_w': f1_w, 'top5': top5, 'auc': auc}

metrics_cnn = compute_metrics(y_true_cnn, y_pred_cnn, y_prob_cnn, 'FaceCNN')
metrics_rn  = compute_metrics(y_true_rn,  y_pred_rn,  y_prob_rn,  'ResNet-50')

# ── Tabela comparativa ────────────────────────────────────────────────────────
print('\n── Comparação ──────────────────────────────────────────')
print(f'{"Métrica":25s} {"FaceCNN":>10s} {"ResNet-50":>10s}')
print('-' * 47)
for key, label in [('acc','Accuracy'), ('f1_mac','F1 Macro'),
                    ('f1_w','F1 Weighted'), ('top5','Top-5 Acc'), ('auc','AUC')]:
    print(f'{label:25s} {metrics_cnn[key]:>9.2f}% {metrics_rn[key]:>9.2f}%')

# ── Confusion Matrix ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, y_true, y_pred, title, cmap in [
    (axes[0], y_true_cnn, y_pred_cnn, 'FaceCNN',   'Blues'),
    (axes[1], y_true_rn,  y_pred_rn,  'ResNet-50', 'Reds'),
]:
    cm = confusion_matrix(y_true, y_pred)
    # Normalizar por linha (recall por classe)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_norm, interpolation='nearest', cmap=cmap, vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title(f'{title} — Confusion Matrix (normalizada)', fontsize=11)
    ax.set_xlabel('Previsto'); ax.set_ylabel('Real')
    # Só mostra labels se poucas classes
    if NUM_CLASSES <= 30:
        ax.set_xticks(range(NUM_CLASSES))
        ax.set_yticks(range(NUM_CLASSES))
        ax.set_xticklabels(full_dataset.classes, rotation=90, fontsize=6)
        ax.set_yticklabels(full_dataset.classes, fontsize=6)

plt.suptitle('Confusion Matrix — FaceCNN vs ResNet-50 (test set)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{BASE}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Per-class accuracy ────────────────────────────────────────────────────────
print('\nPer-class Accuracy (top 10 piores — ResNet-50):')
per_class_rn = confusion_matrix(y_true_rn, y_pred_rn).diagonal() / \
               confusion_matrix(y_true_rn, y_pred_rn).sum(axis=1)
worst = sorted(zip(full_dataset.classes, per_class_rn), key=lambda x: x[1])[:10]
for cls, acc in worst:
    print(f'  {cls}: {acc*100:.1f}%')


# ── ROC Curve (macro OvR) ─────────────────────────────────────────────────────
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

classes = list(range(NUM_CLASSES))
y_bin_cnn = label_binarize(y_true_cnn, classes=classes)
y_bin_rn  = label_binarize(y_true_rn,  classes=classes)

# Macro-average ROC
def macro_roc(y_bin, y_prob):
    all_fpr = np.linspace(0, 1, 200)
    tprs    = []
    for i in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        tprs.append(np.interp(all_fpr, fpr, tpr))
    mean_tpr = np.mean(tprs, axis=0)
    return all_fpr, mean_tpr, auc(all_fpr, mean_tpr)

fpr_cnn, tpr_cnn, auc_cnn = macro_roc(y_bin_cnn, y_prob_cnn)
fpr_rn,  tpr_rn,  auc_rn  = macro_roc(y_bin_rn,  y_prob_rn)

plt.figure(figsize=(8, 6))
plt.plot(fpr_cnn, tpr_cnn, 'b-', linewidth=2, label=f'FaceCNN (AUC = {auc_cnn:.4f})')
plt.plot(fpr_rn,  tpr_rn,  'r-', linewidth=2, label=f'ResNet-50 (AUC = {auc_rn:.4f})')
plt.plot([0,1], [0,1], 'k--', alpha=0.4, label='Random')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Macro OvR (test set)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'AUC FaceCNN:  {auc_cnn:.4f}')
print(f'AUC ResNet-50: {auc_rn:.4f}')

# ── Classification Report completo ────────────────────────────────────────────
print('Classification Report — FaceCNN:')
print(classification_report(y_true_cnn, y_pred_cnn,
      target_names=full_dataset.classes, zero_division=0))

print('\nClassification Report — ResNet-50:')
print(classification_report(y_true_rn, y_pred_rn,
      target_names=full_dataset.classes, zero_division=0))

NameError: name 'RN_LOG' is not defined

---
## 7b. Métricas Detalhadas — F1, Top-5, AUC, ROC, Confusion Matrix

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, top_k_accuracy_score, f1_score
)
import pandas as pd

def get_predictions(model, loader, device):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs  = imgs.to(device)
            out   = model(imgs).float().cpu()
            probs = torch.softmax(out, dim=1)
            all_labels.append(labels)
            all_preds.append(out.argmax(dim=1))
            all_probs.append(probs)
    return (
        torch.cat(all_labels).numpy(),
        torch.cat(all_preds).numpy(),
        torch.cat(all_probs).numpy()
    )

# Carregar melhores modelos
if os.path.exists(CNN_BEST):
    ckpt = torch.load(CNN_BEST, map_location=device)
    cnn_model.load_state_dict(ckpt['model'])
    print(f'✓ CNN carregada: {CNN_BEST}')
else:
    print(f'⚠️  {CNN_BEST} não encontrado — a usar pesos atuais')

if os.path.exists(RN_BEST):
    ckpt = torch.load(RN_BEST, map_location=device)
    resnet_model.load_state_dict(ckpt['model'])
    print(f'✓ ResNet carregada: {RN_BEST}')
else:
    print(f'⚠️  {RN_BEST} não encontrado — a usar pesos atuais')

y_true_cnn, y_pred_cnn, y_prob_cnn = get_predictions(cnn_model,    test_loader, device)
y_true_rn,  y_pred_rn,  y_prob_rn  = get_predictions(resnet_model, test_loader, device)

def compute_metrics(y_true, y_pred, y_prob, name):
    acc    = (y_true == y_pred).mean() * 100
    f1_mac = f1_score(y_true, y_pred, average='macro',    zero_division=0) * 100
    f1_w   = f1_score(y_true, y_pred, average='weighted', zero_division=0) * 100
    top5   = top_k_accuracy_score(y_true, y_prob, k=5)   * 100
    auc    = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro') * 100
    print(f'\n── {name} ──────────────────────────────')
    print(f'  Accuracy:        {acc:.2f}%')
    print(f'  F1 Macro:        {f1_mac:.2f}%')
    print(f'  F1 Weighted:     {f1_w:.2f}%')
    print(f'  Top-5 Accuracy:  {top5:.2f}%')
    print(f'  AUC (OvR macro): {auc:.2f}%')
    return {'modelo': name, 'accuracy': round(acc,4), 'f1_macro': round(f1_mac,4),
            'f1_weighted': round(f1_w,4), 'top5_acc': round(top5,4), 'auc': round(auc,4)}

metrics_cnn = compute_metrics(y_true_cnn, y_pred_cnn, y_prob_cnn, 'FaceCNN')
metrics_rn  = compute_metrics(y_true_rn,  y_pred_rn,  y_prob_rn,  'ResNet-50')

print('\n── Comparação ──────────────────────────────────────────')
print(f'{"Métrica":25s} {"FaceCNN":>10s} {"ResNet-50":>10s}')
print('-' * 47)
for key, label in [('accuracy','Accuracy'), ('f1_macro','F1 Macro'),
                    ('f1_weighted','F1 Weighted'), ('top5_acc','Top-5 Acc'), ('auc','AUC')]:
    print(f'{label:25s} {metrics_cnn[key]:>9.2f}% {metrics_rn[key]:>9.2f}%')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, y_true, y_pred, title, cmap in [
    (axes[0], y_true_cnn, y_pred_cnn, 'FaceCNN',   'Blues'),
    (axes[1], y_true_rn,  y_pred_rn,  'ResNet-50', 'Reds'),
]:
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_norm, interpolation='nearest', cmap=cmap, vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title(f'{title} — Confusion Matrix (normalizada)', fontsize=11)
    ax.set_xlabel('Previsto'); ax.set_ylabel('Real')
    if NUM_CLASSES <= 30:
        ax.set_xticks(range(NUM_CLASSES))
        ax.set_yticks(range(NUM_CLASSES))
        ax.set_xticklabels(full_dataset.classes, rotation=90, fontsize=6)
        ax.set_yticklabels(full_dataset.classes, fontsize=6)
plt.suptitle('Confusion Matrix — FaceCNN vs ResNet-50 (test set)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{BASE}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nPer-class Accuracy (top 10 piores — ResNet-50):')
cm_rn     = confusion_matrix(y_true_rn, y_pred_rn)
per_class_acc = cm_rn.diagonal() / cm_rn.sum(axis=1)
worst = sorted(zip(full_dataset.classes, per_class_acc), key=lambda x: x[1])[:10]
for cls, acc in worst:
    print(f'  {cls}: {acc*100:.1f}%')

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np

classes   = list(range(NUM_CLASSES))
y_bin_cnn = label_binarize(y_true_cnn, classes=classes)
y_bin_rn  = label_binarize(y_true_rn,  classes=classes)

def macro_roc(y_bin, y_prob):
    all_fpr = np.linspace(0, 1, 200)
    tprs = []
    for i in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        tprs.append(np.interp(all_fpr, fpr, tpr))
    mean_tpr = np.mean(tprs, axis=0)
    return all_fpr, mean_tpr, auc(all_fpr, mean_tpr)

fpr_cnn, tpr_cnn, auc_cnn = macro_roc(y_bin_cnn, y_prob_cnn)
fpr_rn,  tpr_rn,  auc_rn  = macro_roc(y_bin_rn,  y_prob_rn)

plt.figure(figsize=(8, 6))
plt.plot(fpr_cnn, tpr_cnn, 'b-', linewidth=2, label=f'FaceCNN (AUC = {auc_cnn:.4f})')
plt.plot(fpr_rn,  tpr_rn,  'r-', linewidth=2, label=f'ResNet-50 (AUC = {auc_rn:.4f})')
plt.plot([0,1], [0,1], 'k--', alpha=0.4, label='Random')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Macro OvR (test set)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'AUC FaceCNN:   {auc_cnn:.4f}')
print(f'AUC ResNet-50: {auc_rn:.4f}')

In [ ]:
print('Classification Report — FaceCNN:')
print(classification_report(y_true_cnn, y_pred_cnn,
      target_names=full_dataset.classes, zero_division=0))
print('\nClassification Report — ResNet-50:')
print(classification_report(y_true_rn, y_pred_rn,
      target_names=full_dataset.classes, zero_division=0))

In [ ]:
import subprocess
from sklearn.metrics import precision_recall_fscore_support

# ── 1. Métricas globais ───────────────────────────────────────────────────────
global_df = pd.DataFrame([metrics_cnn, metrics_rn])
global_df['auc_roc'] = [round(auc_cnn*100, 4), round(auc_rn*100, 4)]
global_df.to_csv(f'{BASE}/metrics_global.csv', index=False)
print('✓ metrics_global.csv')
print(global_df.to_string(index=False))

# ── 2. Métricas por classe ────────────────────────────────────────────────────
rows = []
for modelo, y_true, y_pred in [
    ('FaceCNN',   y_true_cnn, y_pred_cnn),
    ('ResNet-50', y_true_rn,  y_pred_rn),
]:
    prec, rec, f1, sup = precision_recall_fscore_support(y_true, y_pred, zero_division=0)
    cm_d = confusion_matrix(y_true, y_pred).diagonal()
    cm_s = confusion_matrix(y_true, y_pred).sum(axis=1)
    for i, cls in enumerate(full_dataset.classes):
        rows.append({
            'modelo': modelo, 'classe': cls,
            'precision': round(prec[i]*100, 2),
            'recall':    round(rec[i]*100,  2),
            'f1':        round(f1[i]*100,   2),
            'support':   int(sup[i]),
            'correct':   int(cm_d[i]),
            'total':     int(cm_s[i]),
        })
per_class_df = pd.DataFrame(rows)
per_class_df.to_csv(f'{BASE}/metrics_per_class.csv', index=False)
print('\n✓ metrics_per_class.csv')

# ── 3. Histórico de treino ────────────────────────────────────────────────────
with open(CNN_LOG) as f: cnn_h = json.load(f)
with open(RN_LOG)  as f: rn_h  = json.load(f)
cnn_df_h = pd.DataFrame(cnn_h); cnn_df_h.insert(0, 'modelo', 'FaceCNN')
rn_df_h  = pd.DataFrame(rn_h);  rn_df_h.insert(0,  'modelo', 'ResNet-50')
pd.concat([cnn_df_h, rn_df_h], ignore_index=True).to_csv(
    f'{BASE}/training_history.csv', index=False)
print('✓ training_history.csv')

# ── 4. Publicar como Kaggle Dataset (persistente) ────────────────────────────
results_dir = f'{BASE}/results_v8'
os.makedirs(results_dir, exist_ok=True)
import shutil
for f_name in ['metrics_global.csv', 'metrics_per_class.csv', 'training_history.csv']:
    shutil.copy(f'{BASE}/{f_name}', f'{results_dir}/{f_name}')

dataset_meta = {
    'title': 'RNAAPIA Results v8',
    'id': 'goncalojesus/rnaapia-results-v8',
    'licenses': [{'name': 'CC0-1.0'}]
}
with open(f'{results_dir}/dataset-metadata.json', 'w') as f:
    json.dump(dataset_meta, f)

result = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p', results_dir, '--dir-mode', 'zip'],
    capture_output=True, text=True)
print(f'\n✓ Kaggle Dataset: {result.stdout.strip()}')
if result.stderr:
    print(f'  Info: {result.stderr.strip()}')

print(f'\n📁 CSVs em {BASE}:')
print('  metrics_global.csv    — accuracy, F1, Top-5, AUC por modelo')
print('  metrics_per_class.csv — precision, recall, F1 por classe')
print('  training_history.csv  — loss e accuracy por epoch')

---
## 8. Fine-Tuning com Dataset Organizacional
**Antes de correr esta secção**, coloca as fotos em `/kaggle/working/org_dataset/`:
```
org_dataset/
    pessoa1/   ← 5–10 fotos .jpg
    pessoa2/
    ...
```

In [22]:
people = [d for d in os.listdir(PATHS['org_dataset'])
          if os.path.isdir(os.path.join(PATHS['org_dataset'], d))]

if not people:
    print(f'⚠️  Sem pessoas em {PATHS["org_dataset"]}')
    print('   Adiciona as fotos e corre esta secção novamente.')
else:
    print(f'Pessoas encontradas: {people}')
    for p in people:
        imgs = (glob.glob(os.path.join(PATHS['org_dataset'], p, '*.jpg')) +
                glob.glob(os.path.join(PATHS['org_dataset'], p, '*.png')))
        print(f'  {p}: {len(imgs)} imagens')

⚠️  Sem pessoas em /kaggle/working/org_dataset
   Adiciona as fotos e corre esta secção novamente.


In [ ]:
# Alinhar faces org com MTCNN
for person in tqdm(people, desc='MTCNN org'):
    out_dir = os.path.join(PATHS['org_aligned'], person)
    os.makedirs(out_dir, exist_ok=True)
    if len(os.listdir(out_dir)) > 5:
        print(f'  {person}: já processado'); continue
    img_paths = (glob.glob(os.path.join(PATHS['org_dataset'], person, '*.jpg')) +
                 glob.glob(os.path.join(PATHS['org_dataset'], person, '*.png')))
    saved = 0
    for img_path in img_paths:
        try:
            face = mtcnn(Image.open(img_path).convert('RGB'))
            if face is not None:
                Image.fromarray(face.permute(1,2,0).byte().numpy()).save(
                    os.path.join(out_dir, os.path.basename(img_path)))
                saved += 1
        except Exception: pass
    print(f'  {person}: {saved} faces alinhadas')

In [ ]:
# Data augmentation agressiva (~20x por imagem)
aug_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.4, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.3),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
])

AUG_DIR    = f'{BASE}/org_augmented'
N_AUGMENTS = 20
os.makedirs(AUG_DIR, exist_ok=True)

for person in people:
    out_dir = os.path.join(AUG_DIR, person)
    os.makedirs(out_dir, exist_ok=True)
    aligned = (glob.glob(os.path.join(PATHS['org_aligned'], person, '*.jpg')) +
               glob.glob(os.path.join(PATHS['org_aligned'], person, '*.png')))
    for i, img_path in enumerate(aligned):
        img = Image.open(img_path).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
        img.save(os.path.join(out_dir, f'orig_{i:03d}.jpg'))
        for j in range(N_AUGMENTS):
            aug_transform(img).save(os.path.join(out_dir, f'aug_{i:03d}_{j:02d}.jpg'))
    print(f'  {person}: {len(aligned)} originais → {len(os.listdir(out_dir))} total')

In [ ]:
ft_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

org_full        = datasets.ImageFolder(AUG_DIR, transform=ft_transform)
ORG_NUM_CLASSES = len(org_full.classes)
n_ft_train = int(0.70 * len(org_full))
n_ft_val   = int(0.15 * len(org_full))
n_ft_test  = len(org_full) - n_ft_train - n_ft_val

ft_train, ft_val, ft_test = random_split(
    org_full, [n_ft_train, n_ft_val, n_ft_test],
    generator=torch.Generator().manual_seed(42))

ft_train_loader = DataLoader(ft_train, batch_size=16, shuffle=True,  num_workers=2)
ft_val_loader   = DataLoader(ft_val,   batch_size=16, shuffle=False, num_workers=2)
ft_test_loader  = DataLoader(ft_test,  batch_size=16, shuffle=False, num_workers=2)

print(f'Pessoas: {org_full.classes} | Total: {len(org_full)}')
print(f'Train: {n_ft_train} | Val: {n_ft_val} | Test: {n_ft_test}')

In [ ]:
# Fine-tuning a partir do melhor modelo v7
USE_MODEL = 'cnn'  # 'cnn' ou 'resnet'

if USE_MODEL == 'cnn':
    ft_model = FaceCNN(NUM_CLASSES, EMBEDDING_SIZE, IMAGE_SIZE, DROPOUT).to(device)
    ckpt     = torch.load(CNN_BEST, map_location=device)
else:
    ft_model = ResNet50Face(NUM_CLASSES, EMBEDDING_SIZE, DROPOUT).to(device)
    ckpt     = torch.load(RN_BEST, map_location=device)

ft_model.load_state_dict(ckpt['model'])
ft_model.classifier = nn.Linear(EMBEDDING_SIZE, ORG_NUM_CLASSES).to(device)

# Congelar convolucionais, treinar embedding + classifier
for name, param in ft_model.named_parameters():
    param.requires_grad = any(l in name for l in ['embedding', 'classifier'])

trainable = sum(p.numel() for p in ft_model.parameters() if p.requires_grad) / 1e6
print(f'✓ Fine-tuning {USE_MODEL.upper()} | Treináveis: {trainable:.2f}M | Classes: {org_full.classes}')

In [ ]:
FT_EPOCHS    = 30
ft_optimizer = Adam([p for p in ft_model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4)
ft_scheduler = CosineAnnealingLR(ft_optimizer, T_max=FT_EPOCHS, eta_min=1e-6)
ft_scaler    = torch.amp.GradScaler('cuda')
ft_hist      = []
best_ft_acc  = 0.0

for epoch in range(FT_EPOCHS):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(ft_model, ft_train_loader, ft_optimizer, ft_scaler, device, 0.0)
    val_loss,   val_acc   = evaluate(ft_model, ft_val_loader, device)
    ft_scheduler.step()
    elapsed = time.time() - t0
    ft_hist.append({'epoch': epoch, 'train_loss': round(train_loss,4),
                    'val_loss': round(val_loss,4), 'val_acc': round(val_acc,4)})
    print(f'[FT] {epoch+1:02d}/{FT_EPOCHS} | '
          f'Train: {train_loss:.4f} ({train_acc*100:.1f}%) | '
          f'Val: {val_loss:.4f} ({val_acc*100:.1f}%) | {elapsed:.0f}s')
    if val_acc > best_ft_acc:
        best_ft_acc = val_acc
        save_checkpoint(f'{BASE}/best_finetuned.pth', epoch, ft_model, ft_optimizer, ft_scheduler, ft_hist)
        print(f'  ✓ Melhor fine-tuned guardado ({val_acc*100:.2f}%)')

print(f'\n✓ Fine-tuning concluído! Melhor acc: {best_ft_acc*100:.2f}%')

---
## 9. Avaliação Final — FAR / FRR / EER

In [ ]:
from sklearn.metrics import classification_report

ckpt = torch.load(f'{BASE}/best_finetuned.pth', map_location=device)
ft_model.load_state_dict(ckpt['model'])
ft_model.eval()

all_emb, all_labels = [], []
with torch.no_grad():
    for imgs, labels in ft_test_loader:
        emb = ft_model(imgs.to(device), return_embedding=True)
        all_emb.append(emb.cpu()); all_labels.append(labels)
all_emb    = torch.cat(all_emb)
all_labels = torch.cat(all_labels)

gallery = {}
with torch.no_grad():
    for imgs, labels in ft_train_loader:
        emb = ft_model(imgs.to(device), return_embedding=True).cpu()
        for e, l in zip(emb, labels):
            gallery.setdefault(l.item(), []).append(e)

gallery_means  = {k: F.normalize(torch.stack(v).mean(0).unsqueeze(0), dim=1).squeeze()
                  for k, v in gallery.items()}
gallery_matrix = torch.stack([gallery_means[i] for i in range(ORG_NUM_CLASSES)])
similarities   = all_emb @ gallery_matrix.T
max_scores     = similarities.max(dim=1).values.numpy()
pred_labels    = similarities.argmax(dim=1).numpy()
true_labels    = all_labels.numpy()

genuine_scores  = max_scores[pred_labels == true_labels]
impostor_scores = max_scores[pred_labels != true_labels]

thresholds = np.linspace(0, 1, 1000)
FARs = np.array([(impostor_scores >= t).mean() if len(impostor_scores) > 0 else 0.0 for t in thresholds])
FRRs = np.array([(genuine_scores  <  t).mean() if len(genuine_scores)  > 0 else 0.0 for t in thresholds])
eer_idx = np.argmin(np.abs(FARs - FRRs))
EER     = (FARs[eer_idx] + FRRs[eer_idx]) / 2
EER_t   = thresholds[eer_idx]

print(f'EER:           {EER*100:.2f}%')
print(f'Threshold EER: {EER_t:.3f}')
print(f'FAR @ EER:     {FARs[eer_idx]*100:.2f}%')
print(f'FRR @ EER:     {FRRs[eer_idx]*100:.2f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(thresholds, FARs*100, 'r-', label='FAR (%)', linewidth=2)
axes[0].plot(thresholds, FRRs*100, 'b-', label='FRR (%)', linewidth=2)
axes[0].axvline(EER_t, color='g', linestyle='--', label=f'EER={EER*100:.2f}% @ {EER_t:.2f}')
axes[0].set_xlabel('Threshold'); axes[0].set_ylabel('Rate (%)')
axes[0].set_title('FAR / FRR'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].hist(genuine_scores,  bins=30, alpha=0.6, color='green', label='Genuine',  density=True)
axes[1].hist(impostor_scores, bins=30, alpha=0.6, color='red',   label='Impostor', density=True)
axes[1].axvline(EER_t, color='black', linestyle='--', label='EER threshold')
axes[1].set_xlabel('Cosine Similarity'); axes[1].set_title('Score Distributions')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/far_frr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

torch.save({org_full.classes[k]: v for k, v in gallery_means.items()}, f'{BASE}/gallery.pt')
print('\n✓ gallery.pt guardado')
mask = max_scores >= EER_t
if mask.sum() > 0:
    print('\nClassification Report @ EER threshold:')
    print(classification_report(true_labels[mask], pred_labels[mask],
                                target_names=org_full.classes, zero_division=0))

---
## 10. Script de Inferência Local

In [ ]:
script = f'''# realtime_access_control.py
# pip install torch torchvision facenet-pytorch opencv-python
# Ficheiros: best_finetuned.pth + gallery.pt

import torch, cv2
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from facenet_pytorch import MTCNN
from PIL import Image

CHECKPOINT = "best_finetuned.pth"
GALLERY    = "gallery.pt"
THRESHOLD  = {EER_t:.3f}
IMAGE_SIZE = {IMAGE_SIZE}

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True), nn.MaxPool2d(2))
    def forward(self, x): return self.block(x)

class FaceCNN(nn.Module):
    def __init__(self, num_classes, emb=512, sz=192):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            ConvBlock(3,32), ConvBlock(32,64), ConvBlock(64,128), ConvBlock(128,256))
        fm = sz // 16
        self.embedding = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.5),
            nn.Linear(256*fm*fm, emb), nn.BatchNorm1d(emb), nn.ReLU(inplace=True))
        self.classifier = nn.Linear(emb, num_classes)
    def forward(self, x, return_embedding=False):
        e = self.embedding(self.conv_blocks(x))
        return F.normalize(e, dim=1) if return_embedding else self.classifier(e)

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gallery = torch.load(GALLERY, map_location=device)
people  = list(gallery.keys())
gmat    = torch.stack([gallery[p] for p in people]).to(device)

model = FaceCNN(len(people), sz=IMAGE_SIZE).to(device)
model.load_state_dict(torch.load(CHECKPOINT, map_location=device)["model"])
model.eval()

mtcnn = MTCNN(image_size=IMAGE_SIZE, margin=IMAGE_SIZE//6, device=device)
tf    = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)])

cap = cv2.VideoCapture(0)
print(f"Sistema iniciado ({{device}}). Prima Q para sair.")

while True:
    ret, frame = cap.read()
    if not ret: break
    pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    boxes, probs = mtcnn.detect(pil)
    if boxes is not None:
        for box, prob in zip(boxes, probs):
            if prob is None or prob < 0.9: continue
            face = mtcnn(pil)
            if face is None: continue
            t = tf(Image.fromarray(face.permute(1,2,0).byte().numpy())).unsqueeze(0).to(device)
            with torch.no_grad():
                emb   = model(t, return_embedding=True)
                sims  = (emb @ gmat.T).squeeze()
                score = sims.max().item()
                idx   = sims.argmax().item()
            x1,y1,x2,y2 = [int(b) for b in box]
            if score >= THRESHOLD:
                color, label = (0,255,0), f"PERMITIDO: {{people[idx]}} ({{score:.2f}})"
            else:
                color, label = (0,0,255), f"NEGADO ({{score:.2f}})"
            cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
            cv2.putText(frame,label,(x1,y1-10),cv2.FONT_HERSHEY_SIMPLEX,0.65,color,2)
    cv2.imshow("Controlo de Acesso — RNAAPIA", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"): break

cap.release()
cv2.destroyAllWindows()
'''

with open(f'{BASE}/realtime_access_control.py', 'w') as f:
    f.write(script)
print('✓ Script guardado')
print('1. Descarrega best_finetuned.pth e gallery.pt')
print('2. pip install torch torchvision facenet-pytorch opencv-python')
print('3. python realtime_access_control.py')

---
## 📁 Ficheiros gerados
```
best_cnn_v7.pth           ← melhor FaceCNN v7     ← DESCARREGAR
best_resnet_v7.pth        ← melhor ResNet-50 v7   ← DESCARREGAR
best_finetuned.pth        ← modelo fine-tuned     ← DEMO
gallery.pt                ← embeddings org        ← DEMO
dataset_metadata_v7.json  ← 100 identidades usadas
checkpoints/
│  cnn_v7_latest.pth
│  resnet_v7_latest.pth
logs/
│  cnn_v7_log.json
│  resnet_v7_log.json
comparison_v7.png
far_frr_curves.png
realtime_access_control.py
```

## 🔄 Se a sessão cair
Corre **0 → 3** e depois a secção onde ficaste — checkpoints carregam automaticamente.

## ⚠️ Nota importante sobre o classifier
O v7 tem 100 classes em vez de 50. Os pesos do v5 são carregados nas camadas convolucionais
e de embedding — o classifier é reiniciado porque tem dimensão diferente (512→100 em vez de 512→50).
Isto é correto e esperado.